# KD teacher-student: BEATs → FRCRN + PANNs CNN10 (ReviewKD)

| Nhánh | Đường đi | Trạng thái |
| --- | --- | --- |
| Teacher | **nhiễu thật** → chuẩn hoá RMS → BEATs (nạp pretrain, lớp 3, 6, 9, 12) | đóng băng |
| Student | mixture → FRCRN (đã fine-tune) → n̂ = x − ŝ → log-mel → CNN10 → logits 36 | FRCRN đóng băng, CNN10 + ReviewModule huấn luyện |
| KD | ReviewKD: ABF + HCL giữa 4 stage CNN10 và 4 lớp BEATs | chỉ lúc train |

Đánh giá (balanced accuracy): từng mốc SNR (−5, 0, 5, 10, 15, 20) và trung bình của chúng.

Ablation bằng cờ ở cell (1): `USE_FRCRN`, `PRETRAIN_TEACHER`, `REVIEW_WEIGHT`.
Kế hoạch: `plan_reviewkd_beats_cnn10.md`. Mã tái dùng: `scripts/run_reviewkd.py`, `src/model/reviewkd/`, `src/train/reviewkd_trainer.py`.

`RUN_TRAINING = False` chỉ chạy smoke test nhỏ vào `output/test/`.

In [ ]:
import json
import sys
from dataclasses import replace
from pathlib import Path

import torch
from torch import Tensor, nn

ROOT = Path.cwd()
if ROOT.name == "notebook":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
# (1) Cấu hình thí nghiệm. Đổi MỘT cờ mỗi lần để ablation.
EXPERIMENT = "kd-beat-cnn10"
RUN_TRAINING = False        # False: smoke test nhỏ vào output/test
USE_FRCRN = True            # True: student = FRCRN(đóng băng) -> CNN10; False: CNN10 trên mixture (A0)
PRETRAIN_TEACHER = True     # False: BEATs khởi tạo ngẫu nhiên, đóng băng (B2)
REVIEW_WEIGHT = None        # None: lấy từ reviewkd.yaml; 0.0: baseline CE thuần (A0/A1)
NUM_WORKERS = 8

In [ ]:
# (2) Đường dẫn đầu ra, logger, thiết bị
from src.utils.logger import setup_logger
from src.utils.paths import RunPaths

output_dir = Path("output") if RUN_TRAINING else Path("output") / "test"
run = RunPaths.new(EXPERIMENT if RUN_TRAINING else f"{EXPERIMENT}-smoke", ROOT / output_dir)
run.create()
logger = setup_logger(EXPERIMENT, run.log_file)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logger.info("run %s | device %s | RUN_TRAINING=%s", run.run_id, device, RUN_TRAINING)

In [ ]:
# (3) Cấu hình và kiểm tra lưới: stage 4 của CNN10 phải bằng lưới patch BEATs (8 x 24)
from scripts.run_reviewkd import build_review, build_student, check_grids, read_reviewkd_config, resolve_path
from src.model.reviewkd import ReviewKDConfig

raw = read_reviewkd_config()
review_config = ReviewKDConfig.from_dict(raw["review"])
grid = check_grids(raw)
print("grid (freq, time):", grid, "| teacher layers:", review_config.teacher_layers)

In [ ]:
# (4) Các thành phần mới: student có FRCRN, teacher nhận nhiễu thật
from src.model.baseline import BaselineModel
from src.model.beat import BEATClassifier
from src.model.frcrn import FRCRNSeparator
from src.model.reviewkd import rms_normalize, tokens_to_grid


class FRCRNStudent(nn.Module):
    """mixture -> FRCRN (đóng băng) -> n_hat = x - s_hat -> BaselineModel (log-mel + CNN10)."""

    def __init__(self, separator: FRCRNSeparator, student: BaselineModel) -> None:
        super().__init__()
        self.separator = separator.freeze()
        self.student = student
        self.config = student.config

    def train(self, mode: bool = True) -> "FRCRNStudent":
        super().train(mode)
        self.separator.eval()
        return self

    def forward(self, waveform: Tensor, return_features: bool = False):
        with torch.no_grad():
            noise_hat = self.separator(waveform)  # FRCRNSeparator.forward đã trả mixture - speech_hat
        return self.student(noise_hat, return_features=return_features)

    def predict_proba(self, waveform: Tensor) -> Tensor:
        return self(waveform).logits.softmax(dim=-1)


class NoiseTeacher(nn.Module):
    """nhiễu thật -> chuẩn hoá RMS -> BEATs -> 4 lưới (4, B, 768, 8, 24). Luôn eval, no_grad."""

    def __init__(self, beats: BEATClassifier, layers: tuple[int, ...], target_rms: float) -> None:
        super().__init__()
        if target_rms <= 0:
            raise ValueError(f"target_rms must be positive, got {target_rms}")
        self.beats = beats.freeze()
        self.layers = tuple(layers)
        self.target_rms = target_rms
        self.freq_bins = beats.frontend.num_mel_bins // beats.config.input_patch_size

    def train(self, mode: bool = True) -> "NoiseTeacher":
        return super().train(False)

    @torch.no_grad()
    def forward(self, noise: Tensor) -> Tensor:
        output = self.beats(rms_normalize(noise, self.target_rms), layers=self.layers)
        return torch.stack([tokens_to_grid(hidden, self.freq_bins) for hidden in output.hidden])


def build_noise_teacher(pretrain: bool) -> NoiseTeacher:
    beats = BEATClassifier.from_yaml(ROOT / "config" / "beat.yaml")
    if pretrain:
        beats.load_pretrained(resolve_path(raw["teacher"]["beats_checkpoint"]))
    return NoiseTeacher(beats, review_config.teacher_layers, review_config.target_rms).to(device)


def build_frcrn_student(use_frcrn: bool):
    student = build_student(raw)
    if not use_frcrn:
        return student
    separator = FRCRNSeparator.from_yaml(ROOT / "config" / "frcrn.yaml")
    separator.load_pretrained(resolve_path(raw["teacher"]["frcrn_checkpoint"]))
    return FRCRNStudent(separator, student)

In [ ]:
# (5) Trainer: teacher ăn nhiễu thật của batch; chỉ cập nhật tham số requires_grad
import itertools

from src.train import ReviewKDTrainer


class NoiseKDTrainer(ReviewKDTrainer):
    @property
    def trainable_parameters(self):
        parameters = itertools.chain(self.model.parameters(), self.review.parameters())
        return (p for p in parameters if p.requires_grad)

    def training_step(self, batch):
        self._noise = batch["noise"].to(self.device)
        return super().training_step(batch)

    def _teacher_features(self, waveform):
        return super()._teacher_features(self._noise)

In [ ]:
# (6) Chạy một thí nghiệm theo các cờ ở cell (1)
from scripts.finetune import SMOKE_BATCH, SMOKE_SAMPLES
from scripts.load_dataset import Mixture36Dataset, build_dataloader, subsample
from scripts.run_baselines import best_epoch_scores, count_millions
from src.train import ReviewKDTrainConfig
from src.utils.seed import set_seed

SNR_METRICS = tuple(f"balanced_snr_{snr}" for snr in (-5, 0, 5, 10, 15, 20)) + ("macro_acc",)


def run_experiment(name: str) -> dict:
    train_config = ReviewKDTrainConfig.from_dict(raw["train"])
    if REVIEW_WEIGHT is not None:
        train_config = replace(train_config, review_weight=REVIEW_WEIGHT)
    max_samples, workers = None, NUM_WORKERS
    if not RUN_TRAINING:
        train_config = replace(train_config, epochs=1, patience=1, batch_size=SMOKE_BATCH,
                               eval_batch_size=SMOKE_BATCH, review_warmup_epochs=0)
        max_samples, workers = SMOKE_SAMPLES, 0

    set_seed(int(raw["train"].get("seed", 42)))
    duration = raw["dataset"]["duration"]
    train_ds = Mixture36Dataset(split="train", duration=duration, load_noise=True)
    val_ds = Mixture36Dataset(split="validation", duration=duration)
    loaders = {
        "train": build_dataloader(subsample(train_ds, max_samples), train_config.batch_size, True, workers),
        "validation": build_dataloader(subsample(val_ds, max_samples), train_config.eval_batch_size, False, workers),
    }

    model = build_frcrn_student(USE_FRCRN).to(device)
    review = build_review(raw, review_config, grid)
    teacher = build_noise_teacher(PRETRAIN_TEACHER) if train_config.review_weight > 0 else None
    checkpoint_path = run.checkpoint_path(name)
    checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
    logger.info("%s | frcrn=%s | teacher pretrain=%s | lambda=%.3f | model %.2fM | review %.2fM",
                name, USE_FRCRN, PRETRAIN_TEACHER, train_config.review_weight,
                count_millions(model), count_millions(review))

    trainer = NoiseKDTrainer(model, review, teacher, train_config, device, logger, name, checkpoint_path)
    set_seed(int(raw["train"].get("seed", 42)))
    fit = trainer.fit(loaders["train"], loaders["validation"])
    scores = best_epoch_scores(fit.history, fit.best_epoch)
    row = {"model": name, "best_epoch": fit.best_epoch, "epochs_run": fit.epochs_run,
           **{metric: scores[metric] for metric in SNR_METRICS}}
    run.history_path(name).parent.mkdir(parents=True, exist_ok=True)
    run.history_path(name).write_text(json.dumps({"row": row, "history": fit.history}, indent=2), encoding="utf-8")
    return row

In [ ]:
# (7) run_all: RUN_TRAINING = True chạy huấn luyện đầy đủ; False chạy smoke test nhỏ (64 mẫu, 1 epoch, vào output/test).
# Chỉ chạy cell này khi đã đồng ý dùng máy.
import pandas as pd


def run_all() -> pd.DataFrame:
    name = "-".join([
        "frcrn" if USE_FRCRN else "mix",
        "kd" if (REVIEW_WEIGHT is None or REVIEW_WEIGHT > 0) else "ce",
        "pre" if PRETRAIN_TEACHER else "rand",
    ])
    row = run_experiment(name)
    return pd.DataFrame([row]).T.rename(columns={0: "value"}).loc[list(SNR_METRICS)]


run_all()